# API արագության սահմանափակիչ

*1991 Academy · Առաքելություններ*

Քո API-ն թույլ է տալիս `max_calls` հարցում ամեն `window_ms` միլիվայրկյանում ամեն հաճախորդի համար — պահապանը, որ չարաշահումը դարձնում է քաղաքավարի `429`։

Գրի՛ր `create_limiter(max_calls, window_ms)`, որ վերադարձնում է `allow(t)` ֆունկցիա (t = ժամանակադրոշմ մվ-ով)։ Այն վերադարձնում է `True`, եթե `(t - window_ms, t]`-ում տեղի է ունեցել `max_calls`-ից քիչ *թույլատրված* կանչ, այլապես `False`։ Մերժված կանչերը չեն հաշվվում պատուհանի դեմ։ Ժամանակադրոշմերի հերթը — ժամկետանցները ջնջելով առջևից — դասական սահող-պատուհան իրականացումն է։

> 💾 **Աշխատանքդ պահվում է.** Colab-ում վերբեռնված (**File → Upload notebook**) այս նոթբուքը գտնվում է քո Google Drive-ում՝ *Colab Notebooks* թղթապանակում։ Կարող ես փակել այն և շարունակել ավելի ուշ։
>
> ✅ Երբ բոլոր թեստերն անցնեն, վերադարձի՛ր 1991 Academy և սեղմի՛ր **Ես լուծեցի ✓**։

## Քո կոդը

Գրի՛ր լուծումդ ներքևի բջջում և գործարկի՛ր այն. սեղմի՛ր ▶ կամ Shift+Enter։

In [ ]:
# Sliding window rate limiter.
# allow(t): True if fewer than max_calls ALLOWED calls
# happened in the window (t - window_ms, t].
def create_limiter(max_calls, window_ms):
    # keep allowed timestamps in a queue
    def allow(t):
        return True
    return allow

## Ստուգի՛ր լուծումդ

Գործարկի՛ր ներքևի բջիջը։ Եթե թեստը չանցնի, ուղղի՛ր կոդդ, նորից գործարկի՛ր կոդի բջիջը, հետո՝ այս մեկը։

In [ ]:
#@title ▶ Գործարկել թեստերը { display-mode: "form" }
import json, math, traceback
MSG = {
 "all_pass": "🎉 Բոլոր {n} թեստերն անցան։ Վերադարձի՛ր 1991 Academy և սեղմի՛ր «Ես լուծեցի ✓»։",
 "some_pass": "{p}/{n} թեստ է անցնում։ Շարունակի՛ր։",
 "none_ran": "Ոչ մի թեստ չաշխատեց։ Ֆունկցիայիդ անունը {fn} է՞։",
 "expected": "սպասվում էր {e}, ստացվեց {a}",
 "error": "💥 Թեստերը կանգ առան սխալով՝",
 "not_run": "Գործարկե՞լ ես վերևի կոդի բջիջը։ Թեստերը տեսնում են միայն գործարկված կոդը։"
}

def _report(results, error, fn):
    for name, ok, expected, actual in results:
        print(("✓ " if ok else "✗ ") + name + ("" if ok else "   " + MSG["expected"].format(e=expected, a=actual)))
    if error:
        print("\n" + MSG["error"])
        print(error.rstrip())
    passed = sum(1 for r in results if r[1])
    print()
    if results and passed == len(results) and not error:
        print(MSG["all_pass"].format(n=passed))
    elif results:
        print(MSG["some_pass"].format(p=passed, n=len(results)))
    elif not error:
        print(MSG["none_ran"].format(fn=fn))

def _academy_run_tests():
    results = []

    def __check(name, actual, expected):
        try:
            ok = bool(actual == expected)
        except Exception:
            ok = False
        results.append((str(name), ok, repr(expected), repr(actual)))

    def tests():
        allow = create_limiter(3, 1000)
        __check("first three pass", [allow(0), allow(100), allow(200)], [True, True, True])
        __check("fourth inside window blocked", allow(300), False)
        __check("still blocked at window edge", allow(999), False)
        __check("oldest expires, allowed again", allow(1001), True)
        a2 = create_limiter(1, 100)
        __check("independent limiter state", a2(0), True)
        __check("blocked inside its window", a2(50), False)
        __check("free after the window passes", a2(151), True)

    error = None
    try:
        tests()
    except Exception as exc:
        error = traceback.format_exc(limit=-3)
        if isinstance(exc, NameError):
            error += "\n" + MSG["not_run"]
    _report(results, error, 'create_limiter')

_academy_run_tests()

## Հուշումներ

<details><summary>Հուշում 1</summary>

Պահի՛ր թույլատրված ժամանակադրոշմերը հերթում։ Ամեն allow(t)-ի ժամանակ. նախ առջևից հեռացրո՛ւ t - window_ms-ից փոքր կամ հավասար ժամանակադրոշմերը, հետո հերթի երկարությունը համեմատի՛ր max_calls-ի հետ։

</details>

<details><summary>Հուշում 2</summary>

Ավելացրո՛ւ t-ն միայն, երբ կանչը թույլատրված է — մերժված հարցումները երբեք չեն միանում հերթին։

</details>

<details><summary>Հուշում 3</summary>

create_limiter-ում ստեղծված collections.deque, որը allow()-ը կարդում ու փոխում է. popleft() արա ժամկետանց ժամանակադրոշմները, append(t)՝ թույլ տալիս։ Յուրաքանչյուր սահմանափակիչ ստանում է իր սեփական deque-ն։

</details>